# Train YOLOv8n-pose Card Detector

This notebook trains a YOLOv8n-pose model to detect Pokemon cards and predict 4 corner keypoints.

**Steps:**
1. Upload dataset zip
2. Install dependencies
3. Train model (~1-2 hours on T4 GPU)
4. Export to ONNX
5. Download model

**Runtime:** GPU (T4) recommended. Go to Runtime > Change runtime type > GPU.

## 1. Upload Dataset

First, create the dataset zip on your local machine:
```bash
cd data/yolo_card_dataset
zip -r ../../yolo_card_dataset.zip images/ labels/ dataset.yaml
```

Or on Windows PowerShell:
```powershell
Compress-Archive -Path data\yolo_card_dataset\* -DestinationPath yolo_card_dataset.zip
```

Then upload it below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload yolo_card_dataset.zip

In [ ]:
!mkdir -p /content/dataset
!unzip -q yolo_card_dataset.zip -d /content/dataset
!ls /content/dataset/
!echo '---'
!cat /content/dataset/dataset.yaml

In [ ]:
# Fix paths in dataset.yaml for Colab
import yaml

yaml_path = '/content/dataset/dataset.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path'] = '/content/dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('Updated dataset.yaml:')
!cat /content/dataset/dataset.yaml

In [ ]:
# Verify dataset
import os
train_imgs = len(os.listdir('/content/dataset/images/train'))
val_imgs = len(os.listdir('/content/dataset/images/val'))
print(f'Train: {train_imgs} images')
print(f'Val:   {val_imgs} images')

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 3. Train Model

Training YOLOv8n-pose with 4 corner keypoints.
Expected time: ~1-2 hours on T4 GPU for 100 epochs.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n-pose (smallest pose model)
model = YOLO('yolov8n-pose.pt')

# Train
results = model.train(
    data='/content/dataset/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    patience=20,
    workers=2,
    project='runs/pose',
    name='card_detector',
    # Pose loss
    pose=12.0,
    kobj=1.0,
    # Augmentation (dataset already has augmentation)
    hsv_h=0.01,
    hsv_s=0.3,
    hsv_v=0.3,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

In [ ]:
# View training curves
from IPython.display import Image, display
display(Image(filename='runs/pose/card_detector/results.png', width=800))

## 4. Validate

In [ ]:
# Validate on val set
model = YOLO('runs/pose/card_detector/weights/best.pt')
metrics = model.val(
    data='/content/dataset/dataset.yaml',
    device=0,
)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
# Test on a few val images
import os
from IPython.display import Image as IPImage, display

val_dir = '/content/dataset/images/val'
val_imgs = sorted(os.listdir(val_dir))[:5]

for img_name in val_imgs:
    results = model.predict(
        os.path.join(val_dir, img_name),
        save=True,
        project='runs/pose',
        name='val_preview',
        exist_ok=True,
    )

# Show predictions
pred_dir = 'runs/pose/val_preview'
for f in sorted(os.listdir(pred_dir))[:5]:
    if f.endswith(('.jpg', '.png')):
        display(IPImage(filename=os.path.join(pred_dir, f), width=400))

## 5. Export to ONNX

In [ ]:
# Export best model to ONNX
model = YOLO('runs/pose/card_detector/weights/best.pt')
onnx_path = model.export(format='onnx', imgsz=640, simplify=True)
print(f'\nExported: {onnx_path}')

import os
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f'Size: {size_mb:.1f} MB')

In [ ]:
# Quick ONNX validation
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(onnx_path)
inp = sess.get_inputs()[0]
print(f'Input: {inp.name} {inp.shape} {inp.type}')

# Test inference
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
out = sess.run(None, {inp.name: dummy})
print(f'Output shape: {out[0].shape}')
print('ONNX validation OK!')

## 6. Download Model

Download both the .pt and .onnx files.

Place them in your project:
```
models/yolo_card/best.pt
models/yolo_card/card_detector.onnx
```

In [ ]:
from google.colab import files

# Download best.pt
files.download('runs/pose/card_detector/weights/best.pt')

# Download ONNX
files.download(onnx_path)